# 05 · Modelo final: dispersão por região e filtro espacial

Ponto de partida: `bb_backward`, o beta-binomial escolhido pela regra do 1-EP no notebook 03.
Duas extensões são testadas, cada uma mantida só se melhorar seu diagnóstico **e** o erro médio
por UF no LOSO ficar no limite abaixo. As regras estão escritas antes dos resultados, em cada Parte.

Tabelas em `results/tables/final_*`, figuras em `results/figures/final_*.html`.

In [1]:
import sys, json, warnings, time
from pathlib import Path
from dataclasses import asdict
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import get_context
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "notebooks"))
from src import especificacoes as E, modelos as M, espacial as S
import graficos_diagnostico as G
import geopandas as gpd
pd.options.display.float_format = "{:,.4f}".format
T0 = time.time()
RECALCULAR = False   # True: refaz a validação cruzada das extensões
TAB, FIG, CV = ROOT / "results/tables", ROOT / "results/figures", ROOT / "results/cv"
for p in (TAB, FIG, CV): p.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(ROOT / "data/processed/municipios.parquet").reset_index(drop=True)
mapa = E.MAPA_2026
ids = df["id_municipio"].astype(str)
geo = pd.read_parquet(ROOT / "data/processed/municipios_geo.parquet")
geo["id_municipio"] = geo["id_municipio"].astype(str)
geo = geo.set_index("id_municipio").loc[ids].reset_index()
gdf = gpd.GeoDataFrame(geo, geometry=gpd.GeoSeries.from_wkb(geo["geometry"]), crs="EPSG:4674")
ARQ_EV = ROOT / "data/processed/autovetores_esf.parquet"
if ARQ_EV.exists():
    ev = pd.read_parquet(ARQ_EV); ev["id_municipio"] = ev["id_municipio"].astype(str)
else:
    ev_df, ev_I = S.autovetores_esf(gdf, k=100)
    ev = ev_df.reset_index(drop=True).assign(id_municipio=ids.to_numpy())
    ev[["id_municipio"] + list(ev_df.columns)].to_parquet(ARQ_EV)
    pd.DataFrame({"autovetor": ev_df.columns, "I_moran": np.asarray(ev_I)}).to_csv(TAB / "final_esf_moran_autovetores.csv", index=False)
ev = ev.set_index("id_municipio").loc[ids].reset_index(drop=True)
df = pd.concat([df, ev], axis=1)
E.registrar_backward(json.loads((TAB / "selecao_variaveis.json").read_text())["backward_bic"])
E.definir_dados_cv(df, mapa)
y, n = df[mapa["y"]].to_numpy(float), df[mapa["n"]].to_numpy(float)
uf, reg = df[mapa["uf"]].astype(str).to_numpy(), df[mapa["regiao"]].astype(str).to_numpy()
UFS_ = sorted(set(uf)); REGS = sorted(set(reg))
W = S.pesos_queen(gdf)
BASE = E.CANDIDATOS["bb_backward"]
print(len(df), "municípios;", ev.shape[1], "autovetores;", "termos de bb_backward:", len(BASE.termos.vars), "contínuas")

5570 municípios; 100 autovetores; termos de bb_backward: 12 contínuas


## Parte 14 · Dispersão do beta-binomial por região

**Regra (escrita antes do resultado).** Opções de dispersão z ∈ {`const`, `regiao`, `regiao_logn`},
ajustadas a todos os dados com os 12 termos de `bb_backward` (log θ = Zγ). Escolha pelo menor BIC.
A opção escolhida é **aceita** se (1) o teste da razão de verossimilhanças contra `const` tem
p < 0,01 **e** (2) o maior |DP − 1| dos resíduos quantílicos por região é menor que com `const`.
Resíduos: quantílicos aleatorizados, semente 0, como no notebook 02. Falhando, a dispersão fica `const`.

In [2]:
OPCOES = ["const", "regiao", "regiao_logn"]
AJ14, PI14, TH14, R14 = {}, {}, {}, {}
for o in OPCOES:
    if o == "const":
        c = BASE
    else:
        c = E.registrar_extensao(f"bb_disp_{o}", disp=o)
    a = c.ajustar(df, mapa); AJ14[o] = a
    PI14[o] = np.asarray(c.prever(a, df, mapa), float)
    TH14[o] = E.prever_theta(a, df, mapa)
    R14[o] = G.residuos_quantilicos_rapido(y, n, PI14[o], theta=TH14[o], rng=0)
nobs = len(df)
L = []
for o in OPCOES:
    a = AJ14[o]; ll, k = a.loglik, a.k
    st = max(0.0, 2 * (ll - AJ14["const"].loglik)); gl = k - AJ14["const"].k
    L.append({"opcao": o, "llf": ll, "k": k, "AIC": 2 * k - 2 * ll, "BIC": k * np.log(nobs) - 2 * ll,
              "LRT_stat": st if gl else np.nan, "LRT_gl": gl if gl else np.nan,
              "LRT_p": stats.chi2.sf(st, gl) if gl else np.nan})
DISP_T = pd.DataFrame(L); DISP_T.to_csv(TAB / "final_dispersao.csv", index=False)
DP14 = pd.DataFrame({o: pd.Series(R14[o]).groupby(reg).std(ddof=1) for o in OPCOES}).T
DP14["max_abs_dp_menos_1"] = (DP14[REGS] - 1).abs().max(axis=1)
DP14.rename_axis("opcao").to_csv(TAB / "final_dp_regiao.csv")
print(DISP_T.to_string(index=False)); print("\nDP do resíduo quantílico por região"); print(DP14.to_string())

      opcao          llf  k         AIC         BIC  LRT_stat  LRT_gl  LRT_p
      const -36,311.1750 20 72,662.3500 72,794.8530       NaN     NaN    NaN
     regiao -36,140.7353 24 72,329.4706 72,488.4742  340.8794  4.0000 0.0000
regiao_logn -36,064.6118 25 72,179.2237 72,344.8524  493.1263  5.0000 0.0000

DP do resíduo quantílico por região
             Centro-Oeste  Nordeste  Norte  Sudeste    Sul  max_abs_dp_menos_1
const              0.9609    1.1819 1.4093   0.8004 0.9380              0.4093
regiao             1.0059    1.0311 1.0768   1.0229 1.0243              0.0768
regiao_logn        0.9979    1.0248 1.0952   1.0089 1.0109              0.0952


In [3]:
ESCOLHA14 = DISP_T.sort_values("BIC").iloc[0]["opcao"]
lin = DISP_T.set_index("opcao").loc[ESCOLHA14]
if ESCOLHA14 == "const":
    DISP_OK, MOTIVO14 = False, "const tem o menor BIC; nenhuma dispersão variável escolhida"
else:
    p_ok = bool(lin.LRT_p < 0.01); d_ok = bool(DP14.loc[ESCOLHA14, "max_abs_dp_menos_1"] < DP14.loc["const", "max_abs_dp_menos_1"])
    DISP_OK = p_ok and d_ok
    MOTIVO14 = (f"LRT p={lin.LRT_p:.3g} ({'<' if p_ok else '>='} 0,01); max|DP-1| {DP14.loc['const','max_abs_dp_menos_1']:.3f} (const) -> "
                f"{DP14.loc[ESCOLHA14,'max_abs_dp_menos_1']:.3f} ({ESCOLHA14}) ({'melhora' if d_ok else 'não melhora'})")
DISP = ESCOLHA14 if DISP_OK else "const"
print("escolhida por BIC:", ESCOLHA14, "| aceita:", DISP_OK, "|", MOTIVO14, "| dispersão adotada:", DISP)
fig = go.Figure([go.Bar(name=o, x=REGS, y=DP14.loc[o, REGS].values) for o in OPCOES])
fig.add_hline(y=1, line_dash="dash", line_color="#888")
fig.update_layout(barmode="group", title="DP do resíduo quantílico por região, por opção de dispersão (ideal = 1)", yaxis_title="DP", template="plotly_white", height=420)
G.salvar(fig, FIG / "final_dp_regiao.html"); fig.show()

escolhida por BIC: regiao_logn | aceita: True | LRT p=2.43e-104 (< 0,01); max|DP-1| 0.409 (const) -> 0.095 (regiao_logn) (melhora) | dispersão adotada: regiao_logn


**Leitura.** O BIC escolhe `regiao_logn`, e o LRT rejeita a dispersão constante (p = 2,4e-104). O desvio residual por região fica perto de 1: o máximo |DP − 1| cai de 0,409 (`const`) para 0,095. O LRT supõe municípios independentes; o adendo a seguir refaz o teste com covariância robusta.

*Números da execução de 2026-11-09; se o notebook for reexecutado, confira as saídas acima.*

### Parte 14 (adendo): teste de Wald robusto para a dispersão

Regra adicionada em 2026-11-09, depois dos primeiros resultados do notebook.

O LRT supõe observações independentes e, com resíduos espacialmente correlacionados, rejeita demais. Por isso a escolha da dispersão passa a ser decidida por um **Wald sequencial em γ** com a covariância de Conley do próprio ajuste (corte = regra da Parte 16 aplicada ao correlograma dos resíduos quantílicos desse ajuste, mesmas faixas).

- Passo A: as 4 dummies de região no ajuste `regiao` (contra `const`).
- Passo B: `log_n` (1 gl) no ajuste `regiao_logn`; só é testado se A for mantido.
- Um passo é mantido se p < 0,01 sob Conley ×1 (qui-quadrado). Conley ×0,5, ×2 e cluster por UF são reportados como F com 26 gl no denominador (27 UFs - 1).
- LRT e BIC continuam reportados, mas deixam de decidir sozinhos. Resultado: `const` se A falha; `regiao` se B falha; senão `regiao_logn`.

In [4]:
FAIXAS = [0, 10, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000]
lat, lon = df["lat"].to_numpy(float), df["lon"].to_numpy(float)
def corte_de(cg):
    ab = cg[cg.correlacao < 0.05]
    return float(ab.iloc[0]["ini_km"]) if len(ab) else float(cg.iloc[-1]["fim_km"])
def passo_wald(passo, o, sel):
    a = AJ14[o]; mod = a.modelo; nm = list(a.nomes)
    ind = [i for i, s in enumerate(nm) if sel(s)]; q = len(ind)
    corte = corte_de(S.correlograma(R14[o], lat, lon, FAIXAS))
    covs = {"conley_05": S.cov_conley(mod, lat, lon, corte * 0.5), "conley_1": S.cov_conley(mod, lat, lon, corte),
            "conley_2": S.cov_conley(mod, lat, lon, corte * 2), "cluster_uf": S.cov_cluster(mod, uf, correcao=True)}
    linhas = []
    for tag, cv in covs.items():
        for tipo, gd in ([("chi2", None)] if tag == "conley_1" else []) + [("F", len(UFS_) - 1)]:
            est, gl, p = S.wald_robusto(mod.params, cv, ind, gl_den=gd)
            linhas.append(dict(passo=passo, ajuste=o, termos=";".join(nm[i] for i in ind), covariancia=tag, corte_km=corte * (0.5 if tag == "conley_05" else 2 if tag == "conley_2" else 1) if tag != "cluster_uf" else np.nan,
                               teste=tipo, estat=est, gl=gl, gl_den=gd if gd else np.nan, p=p))
    return pd.DataFrame(linhas), corte
WA, corte_A = passo_wald("A", "regiao", lambda s: s.startswith("disp_reg_"))
A_OK = bool(WA[(WA.covariancia == "conley_1") & (WA.teste == "chi2")].p.iloc[0] < 0.01)
if A_OK:
    WB, corte_B = passo_wald("B", "regiao_logn", lambda s: s == "disp_log_n")
    B_OK = bool(WB[(WB.covariancia == "conley_1") & (WB.teste == "chi2")].p.iloc[0] < 0.01)
    WALD_T = pd.concat([WA, WB], ignore_index=True)
else:
    B_OK, corte_B, WALD_T = False, np.nan, WA
WALD_T["mantido"] = WALD_T.passo.map({"A": A_OK, "B": B_OK})
WALD_T.to_csv(TAB / "final_dispersao_wald.csv", index=False)
print(WALD_T.to_string(index=False))
DISP_ANT, DISP_OK_ANT = DISP, DISP_OK
DISP = "regiao_logn" if (A_OK and B_OK) else "regiao" if A_OK else "const"
DISP_OK = DISP != "const"
pa = WA[(WA.covariancia == "conley_1") & (WA.teste == "chi2")].p.iloc[0]
MOTIVO14 = (f"Wald robusto (Conley x1, chi2): passo A (dummies de região) p={pa:.3g} -> {'mantido' if A_OK else 'descartado'}"
            + (f"; passo B (log_n) p={WB[(WB.covariancia == 'conley_1') & (WB.teste == 'chi2')].p.iloc[0]:.3g} -> {'mantido' if B_OK else 'descartado'}" if A_OK else "")
            + f". Antes (LRT/BIC): {DISP_ANT}")
WALD_DEC = {"dispersao_adotada": DISP, "A_mantido": A_OK, "B_mantido": B_OK, "corte_A_km": corte_A, "corte_B_km": corte_B,
            "tabela": WALD_T.to_dict(orient="records"), "motivo": MOTIVO14}
print("dispersão adotada:", DISP, "|", MOTIVO14)

passo      ajuste                                                              termos covariancia  corte_km teste   estat  gl  gl_den      p  mantido
    A      regiao disp_reg_Centro-Oeste;disp_reg_Nordeste;disp_reg_Norte;disp_reg_Sul   conley_05  100.0000     F 22.3398   4 26.0000 0.0000     True
    A      regiao disp_reg_Centro-Oeste;disp_reg_Nordeste;disp_reg_Norte;disp_reg_Sul    conley_1  200.0000  chi2 56.6271   4     NaN 0.0000     True
    A      regiao disp_reg_Centro-Oeste;disp_reg_Nordeste;disp_reg_Norte;disp_reg_Sul    conley_1  200.0000     F 14.1568   4 26.0000 0.0000     True
    A      regiao disp_reg_Centro-Oeste;disp_reg_Nordeste;disp_reg_Norte;disp_reg_Sul    conley_2  400.0000     F 14.6689   4 26.0000 0.0000     True
    A      regiao disp_reg_Centro-Oeste;disp_reg_Nordeste;disp_reg_Norte;disp_reg_Sul  cluster_uf       NaN     F  8.6058   4 26.0000 0.0001     True
    B regiao_logn                                                          disp_log_n   conley_05  1

**Leitura.** Os dois passos se mantêm em todas as covariâncias. Passo A (dummies de região na dispersão): χ² = 56,6, p = 1,5e-11 com Conley a 200 km; o caso mais fraco é o conglomerado por UF (F = 8,6, p = 1,5e-4). Passo B (log n): χ² = 66,2, p = 4,2e-16; com Conley ×2 (400 km), F = 44,5, p = 4,4e-7. A dependência espacial infla os EPs, mas não a ponto de apagar a dispersão por região e porte: `regiao_logn` fica.

*Números da execução de 2026-11-09; se o notebook for reexecutado, confira as saídas acima.*

## Parte 15 · Filtro espacial de Moran (ESF)

**Regra (escrita antes do resultado).** Entram em X os k primeiros autovetores de Moran (ordenados
por I decrescente; dependem só da geometria, sem vazamento de y), k ∈ {0, 10, 25, 50, 100}, com a
dispersão adotada na Parte 14 se foi aceita (senão `const`). Escolha do k pelo menor BIC.
O filtro é **aceito** se o k escolhido for > 0 e o I de Moran dos resíduos quantílicos (pesos Queen,
semente 0, 999 permutações) cair em relação a k = 0. Tabela por k: llf, nº de parâmetros, BIC, I, p_sim.

In [5]:
KS = [0, 10, 25, 50, 100]
AJ15, R15, L = {}, {}, []
for k in KS:
    c = E.registrar_extensao(f"bb_esf_k{k}", disp=DISP, esf=k)
    a = c.ajustar(df, mapa); AJ15[k] = a
    pi = np.asarray(c.prever(a, df, mapa), float)
    R15[k] = G.residuos_quantilicos_rapido(y, n, pi, theta=E.prever_theta(a, df, mapa), rng=0)
    m = S.moran_residuos(R15[k], W, permutacoes=999, semente=0)
    L.append({"k_esf": k, "dispersao": DISP, "llf": a.loglik, "n_params": a.k, "BIC": a.k * np.log(nobs) - 2 * a.loglik,
              "I_moran": m["I"], "p_sim": m["p_sim"], "dp_residuo": R15[k].std(ddof=1)})
ESF_T = pd.DataFrame(L); ESF_T.to_csv(TAB / "final_esf_k.csv", index=False)
print(ESF_T.to_string(index=False))

 k_esf   dispersao          llf  n_params         BIC  I_moran  p_sim  dp_residuo
     0 regiao_logn -36,064.6118        25 72,344.8524   0.3390 0.0010      1.0204
    10 regiao_logn -35,997.4289        35 72,296.7381   0.3190 0.0010      1.0215
    25 regiao_logn -35,802.7219        50 72,036.7013   0.2712 0.0010      1.0242
    50 regiao_logn -35,631.5361        75 71,909.9584   0.2272 0.0010      1.0298
   100 regiao_logn -35,511.4335       125 72,101.0108   0.1915 0.0010      1.0273


In [6]:
K_ESC = int(ESF_T.sort_values("BIC").iloc[0]["k_esf"])
I0, Ik = float(ESF_T.set_index("k_esf").loc[0, "I_moran"]), float(ESF_T.set_index("k_esf").loc[K_ESC, "I_moran"])
ESF_OK = K_ESC > 0 and Ik < I0
MOTIVO15 = f"k escolhido por BIC = {K_ESC}; I de Moran {I0:.4f} (k=0) -> {Ik:.4f} (k={K_ESC})" + ("" if K_ESC > 0 else "; k=0 vence, sem filtro")
print("ESF aceito (diagnóstico):", ESF_OK, "|", MOTIVO15)
fig = go.Figure(go.Scatter(x=ESF_T.k_esf, y=ESF_T.I_moran, mode="lines+markers", marker=dict(size=9, color="#1b6ca8")))
fig.update_layout(title="I de Moran dos resíduos quantílicos contra o nº de autovetores (k)", xaxis_title="k", yaxis_title="I de Moran", template="plotly_white", height=400)
G.salvar(fig, FIG / "final_moran_k.html"); fig.show()

ESF aceito (diagnóstico): True | k escolhido por BIC = 50; I de Moran 0.3390 (k=0) -> 0.2272 (k=50)


**Leitura.** Como diagnóstico, o ESF funciona: com k = 50 autovetores (escolha do BIC) o I de Moran do resíduo cai de 0,339 para 0,227. A aceitação final depende da validação cruzada por UF, abaixo: os autovetores são da malha inteira, e a UF retirada continua no filtro.

*Números da execução de 2026-11-09; se o notebook for reexecutado, confira as saídas acima.*

### Parte 15 (adendo): covariáveis defasadas espacialmente (SLX)

Regra adicionada em 2026-11-09, depois dos primeiros resultados do notebook.

O I de Moran residual do modelo base é alto (0,339). Como remédio, tenta-se SLX: W·x (vizinhança rainha, linhas padronizadas) entra em X. Com `disp` = dispersão adotada e `esf` = 0, comparam-se três opções: **nenhuma**, **W·lag**, **W·(as 12 variáveis do modelo base)**. Escolha por BIC; reporta-se loglik, nº de parâmetros, BIC e I de Moran dos resíduos quantílicos (mesma conta da Parte 15).

`SLX_OK` = a opção escolhida não é "nenhuma" **e** o I de Moran cai contra "nenhuma". A defasagem usa só covariáveis, conhecidas em todo lugar: é calculada uma vez nos dados completos; a padronização é ajustada só na dobra de treino. A partir daqui `df`/`mapa` são as versões com as colunas `w_*`.

In [7]:
df, mapa = S.adicionar_slx(df, mapa, list(BASE.termos.vars), W)
E.definir_dados_cv(df, mapa)
OPC_SLX = {"nenhuma": (), "W_lag": ("lag",), "W_12vars": tuple(BASE.termos.vars)}
AJ_SLX, L = {}, []
for o, ch in OPC_SLX.items():
    c = E.registrar_extensao(f"bb_slx_{o}", disp=DISP, esf=0, slx=ch)
    a = c.ajustar(df, mapa); AJ_SLX[o] = a
    pi = np.asarray(c.prever(a, df, mapa), float)
    r = G.residuos_quantilicos_rapido(y, n, pi, theta=E.prever_theta(a, df, mapa), rng=0)
    m = S.moran_residuos(r, W, permutacoes=999, semente=0)
    L.append({"opcao": o, "dispersao": DISP, "n_slx": len(ch), "llf": a.loglik, "n_params": a.k, "BIC": a.k * np.log(nobs) - 2 * a.loglik,
              "I_moran": m["I"], "p_sim": m["p_sim"], "dp_residuo": r.std(ddof=1)})
SLX_T = pd.DataFrame(L); SLX_T.to_csv(TAB / "final_slx.csv", index=False)
print(SLX_T.to_string(index=False))
SLX_ESC = SLX_T.sort_values("BIC").iloc[0]["opcao"]
I_none, I_esc = float(SLX_T.set_index("opcao").loc["nenhuma", "I_moran"]), float(SLX_T.set_index("opcao").loc[SLX_ESC, "I_moran"])
SLX_OK = SLX_ESC != "nenhuma" and I_esc < I_none
SLX_CHAVES = OPC_SLX[SLX_ESC] if SLX_OK else ()
MOTIVO_SLX = f"opção por BIC = {SLX_ESC}; I de Moran {I_none:.4f} (nenhuma) -> {I_esc:.4f} ({SLX_ESC})" + ("" if SLX_OK else "; SLX não aceito no diagnóstico")
SLX_DEC = {"opcao_bic": SLX_ESC, "bic": {r.opcao: float(r.BIC) for r in SLX_T.itertuples()}, "moran": {r.opcao: float(r.I_moran) for r in SLX_T.itertuples()},
           "diagnostico_ok": bool(SLX_OK), "chaves": list(SLX_CHAVES)}
print("SLX aceito (diagnóstico):", SLX_OK, "|", MOTIVO_SLX)

   opcao   dispersao  n_slx          llf  n_params         BIC  I_moran  p_sim  dp_residuo
 nenhuma regiao_logn      0 -36,064.6118        25 72,344.8524   0.3390 0.0010      1.0204
   W_lag regiao_logn      1 -36,057.9608        26 72,340.1756   0.3370 0.0010      1.0202
W_12vars regiao_logn     12 -35,913.9678        37 72,147.0663   0.3328 0.0010      1.0245
SLX aceito (diagnóstico): True | opção por BIC = W_12vars; I de Moran 0.3390 (nenhuma) -> 0.3328 (W_12vars)


**Leitura.** O BIC escolhe `W_12vars` (as 12 covariáveis defasadas), com BIC 72.147 contra 72.345 sem SLX. O ganho de ajuste é real, mas o I de Moran do resíduo quase não muda: 0,339 para 0,333. O SLX explica pouco da estrutura espacial; a correlação que resta não vem de covariáveis dos vizinhos.

*Números da execução de 2026-11-09; se o notebook for reexecutado, confira as saídas acima.*

## Aceitação pela validação cruzada

**Regra (escrita antes do resultado).** Candidatos: `bb_disp` (dispersão escolhida, k = 0),
`bb_esf` (`const`, k escolhido) e `bb_final` (as duas adições). Só entram na CV as adições cujo
diagnóstico passou. Cada uma roda em 5-fold e LOSO. Limite = `mae_uf_pp(bb_backward)` + `se_uf_pp` do
melhor modelo LOSO (o EP da regra do 1-EP do notebook 03), lidos de `cv_loso.csv`. Uma adição que
falha o diagnóstico ou o limite é descartada. O modelo final é `bb_backward` mais as adições aceitas
(se as duas passam isoladamente, vale `bb_final` só se ele também passar o limite; senão, a de menor
erro por UF). Informativo: se o final está dentro da faixa de empate original (melhor + EP).

### Validação cruzada (adendo): candidato `bb_slx`

Regra adicionada em 2026-11-09, depois dos primeiros resultados do notebook.

Se `SLX_OK`, entra o candidato `bb_slx` (dispersão adotada, `esf` = 0, SLX escolhido). Aceitação como a do ESF: o I de Moran cai **e** o `mae_uf` LOSO fica no limite (regra de 1 EP, 1,1308 como já calculado). Se `bb_slx` for aceito, `modelo_final` = `bb_slx`; senão a lógica anterior decide.

In [8]:
CAND = {}
if DISP_OK: CAND["bb_disp"] = dict(disp=DISP, esf=0)
if ESF_OK: CAND["bb_esf"] = dict(disp="const", esf=K_ESC)
if DISP_OK and ESF_OK: CAND["bb_final"] = dict(disp=DISP, esf=K_ESC)
if SLX_OK: CAND["bb_slx"] = dict(disp=DISP, esf=0, slx=tuple(SLX_CHAVES))
for nome, kw in CAND.items(): E.registrar_extensao(nome, **kw)   # antes do pool: os workers são forks
folds5, foldsL = M.folds_kfold(len(df), k=5, seed=0), M.folds_por_grupo(uf)
def _cv(nome, esq):
    arq = CV / f"{esq}_{nome}.parquet"
    return pd.read_parquet(arq)["pi"].to_numpy() if arq.exists() and not RECALCULAR else None
def rodar(nomes, esq, folds):
    pend = [k for k in nomes if _cv(k, esq) is None]
    saida = {}
    if pend:
        with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
            fut = {(k, i): ex.submit(E.prever_fold, k, tr, te) for k in pend for i, (tr, te) in enumerate(folds)}
            for k in pend:
                pi = np.full(len(df), np.nan)
                for i, (tr, te) in enumerate(folds): pi[te] = fut[(k, i)].result()
                pd.DataFrame({"pi": pi}).to_parquet(CV / f"{esq}_{k}.parquet"); saida[k] = pi
    for k in nomes:
        if k not in saida: saida[k] = _cv(k, esq)
    return saida
NOMES_CV = ["bb_backward"] + list(CAND)
PI5, PIL = rodar(NOMES_CV, "kfold", folds5), rodar(NOMES_CV, "loso", foldsL)
print("NaN:", {k: int(np.isnan(v).sum()) for d in (PI5, PIL) for k, v in d.items() if np.isnan(v).any()} or "nenhum", "|", round(time.time() - T0), "s")
obs_uf = pd.Series(y).groupby(uf).sum() / pd.Series(n).groupby(uf).sum()
obs_reg = pd.Series(y).groupby(reg).sum() / pd.Series(n).groupby(reg).sum()
def erros(pi, esq, folds):
    a_uf = (M.agregar(pi, n, uf).drop("Brasil") - obs_uf).abs() * 100
    a_rg = (M.agregar(pi, n, reg).drop("Brasil") - obs_reg).abs() * 100
    m = M.metricas(y, n, pi)
    por = [M.metricas(y[te], n[te], pi[te])["mae_pond_pp"] for _, te in folds]
    m.update(mae_uf_pp=a_uf.mean(), se_uf_pp=a_uf.std(ddof=1) / np.sqrt(len(a_uf)), mae_reg_pp=a_rg.mean(), max_reg_pp=a_rg.max(),
             se_mae_pond_pp=np.std(por, ddof=1) / np.sqrt(len(por)))
    return m
LIN = []
for esq, PI, folds in [("5-fold", PI5, folds5), ("LOSO", PIL, foldsL)]:
    for k in NOMES_CV: LIN.append({"esquema": esq, "modelo": k, **erros(PI[k], esq, folds)})
CVT = pd.DataFrame(LIN); CVT.to_csv(TAB / "final_cv.csv", index=False)
cols = ["esquema", "modelo", "mae_uf_pp", "se_uf_pp", "mae_pond_pp", "se_mae_pond_pp", "mae_reg_pp", "rmse_pond_pp", "deviance"]
print(CVT[[c for c in cols if c in CVT]].to_string(index=False))

NaN: nenhum | 205 s
esquema      modelo  mae_uf_pp  se_uf_pp  mae_pond_pp  se_mae_pond_pp  mae_reg_pp  rmse_pond_pp        deviance
 5-fold bb_backward     0.7886    0.1338       1.1588          0.0280      0.2487        1.5325    132,975.0059
 5-fold     bb_disp     0.7975    0.1411       1.1409          0.0330      0.2618        1.5275    132,337.2856
 5-fold      bb_esf     0.5585    0.1252       1.1190          0.0382      0.3281        1.4891    124,279.6224
 5-fold    bb_final     0.5640    0.1311       1.1040          0.0407      0.3131        1.4877    124,101.6710
 5-fold      bb_slx     0.7546    0.1326       1.1511          0.0614      0.3078        1.5208    130,393.3014
   LOSO bb_backward     1.0004    0.1423       1.3147          0.1077      0.3569        1.7049    163,237.8075
   LOSO     bb_disp     0.9951    0.1447       1.2698          0.1116      0.2762        1.6704    157,373.1087
   LOSO      bb_esf     1.4460    0.6445       5.1708          0.6453      2.1062   

In [9]:
loso = pd.read_csv(TAB / "cv_loso.csv").sort_values("mae_uf_pp").reset_index(drop=True)
melhor, se_melhor = float(loso.loc[0, "mae_uf_pp"]), float(loso.loc[0, "se_uf_pp"])
mae_bb = float(loso.set_index("modelo").loc["bb_backward", "mae_uf_pp"])
LIMITE = mae_bb + se_melhor
FAIXA = melhor + se_melhor
Lm = CVT[CVT.esquema == "LOSO"].set_index("modelo")["mae_uf_pp"]
passa = {k: bool(Lm[k] <= LIMITE) for k in CAND}
ACEITA = {"disp": bool(DISP_OK and passa.get("bb_disp", False)), "esf": bool(ESF_OK and passa.get("bb_esf", False))}
if ACEITA["disp"] and ACEITA["esf"] and not passa.get("bb_final", False):
    pior = "esf" if Lm["bb_disp"] <= Lm["bb_esf"] else "disp"; ACEITA[pior] = False
FINAL = {(False, False): "bb_backward", (True, False): "bb_disp", (False, True): "bb_esf", (True, True): "bb_final"}[(ACEITA["disp"], ACEITA["esf"])]
ACEITA["slx"] = bool(SLX_OK and passa.get("bb_slx", False))
if ACEITA["slx"]: FINAL = "bb_slx"
cf = (E.CANDIDATOS.get(FINAL) or E.EXTENSOES[FINAL])
termos = {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(cf.termos).items()}
dec = {"disp": DISP, "k_esf": K_ESC,
       "aceitas": {"disp": ACEITA["disp"], "esf": ACEITA["esf"]},
       "motivos": {"disp": MOTIVO14 + f"; CV LOSO mae_uf={Lm.get('bb_disp', float('nan')):.4f} vs limite {LIMITE:.4f}",
                   "esf": MOTIVO15 + f"; CV LOSO mae_uf={Lm.get('bb_esf', float('nan')):.4f} vs limite {LIMITE:.4f}"},
       "diagnostico_ok": {"disp": bool(DISP_OK), "esf": bool(ESF_OK)},
       "mae_uf_loso": {k: float(v) for k, v in Lm.items()},
       "limite": float(LIMITE), "mae_uf_bb_backward": mae_bb, "se_melhor_loso": se_melhor,
       "faixa_empate_original": float(FAIXA), "final_dentro_da_faixa_original": bool(Lm[FINAL] <= FAIXA),
       "modelo_final": FINAL, "termos_final": termos}
dec["aceitas"]["slx"] = ACEITA["slx"]; dec["diagnostico_ok"]["slx"] = bool(SLX_OK)
dec["motivos"]["slx"] = MOTIVO_SLX + f"; CV LOSO mae_uf={Lm.get('bb_slx', float('nan')):.4f} vs limite {LIMITE:.4f}"
dec["dispersao_wald"] = WALD_DEC; dec["slx"] = SLX_DEC
(TAB / "final_decisao.json").write_text(json.dumps(dec, indent=2, ensure_ascii=False, default=str))
print(json.dumps({k: dec[k] for k in ["disp", "k_esf", "aceitas", "motivos", "limite", "faixa_empate_original", "final_dentro_da_faixa_original", "modelo_final"]}, indent=2, ensure_ascii=False))
print("tempo total:", round(time.time() - T0), "s")

{
  "disp": "regiao_logn",
  "k_esf": 50,
  "aceitas": {
    "disp": true,
    "esf": false,
    "slx": true
  },
  "motivos": {
    "disp": "Wald robusto (Conley x1, chi2): passo A (dummies de região) p=1.48e-11 -> mantido; passo B (log_n) p=4.17e-16 -> mantido. Antes (LRT/BIC): regiao_logn; CV LOSO mae_uf=0.9951 vs limite 1.1308",
    "esf": "k escolhido por BIC = 50; I de Moran 0.3390 (k=0) -> 0.2272 (k=50); CV LOSO mae_uf=1.4460 vs limite 1.1308",
    "slx": "opção por BIC = W_12vars; I de Moran 0.3390 (nenhuma) -> 0.3328 (W_12vars); CV LOSO mae_uf=1.0224 vs limite 1.1308"
  },
  "limite": 1.1307676100284951,
  "faixa_empate_original": 1.0605455808266946,
  "final_dentro_da_faixa_original": true,
  "modelo_final": "bb_slx"
}
tempo total: 205 s


**Leitura.** No LOSO, o limite de aceitação é 1,131 p.p. de MAE por UF. `bb_disp` (0,995) e `bb_slx` (1,022) passam; `bb_esf` (1,446) e `bb_final` (1,412) não, com EP e erro ponderado muito maiores: o filtro espacial extrapola mal para uma UF nova, embora seja o melhor nas 5 dobras (0,559). Pela regra fixada antes, o modelo final é `bb_slx`. O `bb_disp` tem MAE um pouco menor, e a regra não exigia queda mínima do I de Moran; essa limitação está em [07-pendencias](../docs/pressupostos/07-pendencias.md).

*Números da execução de 2026-11-09; se o notebook for reexecutado, confira as saídas acima.*

## Parte 16 · Erros-padrão de Conley (HAC espacial) do modelo final

O modelo final (`modelo_final` de `final_decisao.json`) ainda deixa resíduos correlacionados no espaço
(Moran > 0; o filtro de autovetores foi rejeitado porque falha no LOSO). Os EPs baseados no modelo
supõem observações independentes, logo são pequenos demais. Esta Parte troca-os por EPs robustos.

**Regra (escrita antes de ver os EPs).**
1. Correlograma dos resíduos quantílicos (semente 0) do modelo final, faixas [ini, fim) em km:
   0, 10, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000.
2. **Corte de Conley = borda inferior da primeira faixa com correlação < 0,05.**
3. Sensibilidade: corte × 0,5 e × 2, e conglomerado por UF (27 grupos, com correção G/(G−1)·(N−1)/(N−k)),
   que é um limite conservador (trata cada UF como um bloco com correlação livre dentro dele).
4. Núcleo de Bartlett, distância de círculo máximo, sem correção de pequena amostra.
5. Um coeficiente "muda" se |z| > 1,96 com o EP do modelo e não com o EP de Conley (corte da regra), ou o inverso.

In [10]:
FIN = json.loads((TAB / "final_decisao.json").read_text())
NOME_F = FIN["modelo_final"]
CF = E.CANDIDATOS.get(NOME_F) or E.EXTENSOES[NOME_F]
AF = CF.ajustar(df, mapa)
NB_ = AF.extra["n_beta"]
PIF = np.asarray(CF.prever(AF, df, mapa), float); THF = E.prever_theta(AF, df, mapa)
RF = G.residuos_quantilicos_rapido(y, n, PIF, theta=THF, rng=0)
lat, lon = df["lat"].to_numpy(float), df["lon"].to_numpy(float)
FAIXAS = [0, 10, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000]
CG = S.correlograma(RF, lat, lon, FAIXAS); CG.to_csv(TAB / "final_correlograma.csv", index=False)
abaixo = CG[CG.correlacao < 0.05]
CORTE = float(abaixo.iloc[0]["ini_km"])
print("modelo final:", NOME_F, "| n_beta:", NB_, "| params:", len(AF.modelo.params), "| corte de Conley:", CORTE, "km")
print(CG.to_string(index=False))
fig = go.Figure(go.Bar(x=[f"{a:g}-{b:g}" for a, b in zip(CG.ini_km, CG.fim_km)], y=CG.correlacao, marker_color="#1b6ca8"))
fig.add_hline(y=0.05, line_dash="dash", line_color="#c2571a", annotation_text="limiar 0,05")
fig.update_layout(title=f"Correlograma dos resíduos quantílicos ({NOME_F}); corte = {CORTE:g} km", xaxis_title="faixa de distância (km)", yaxis_title="correlação média", template="plotly_white", height=420)
G.salvar(fig, FIG / "final_correlograma.html"); fig.show()

modelo final: bb_slx | n_beta: 31 | params: 37 | corte de Conley: 200.0 km
  ini_km     fim_km   pares  correlacao
  0.0000    10.0000     617      0.3962
 10.0000    25.0000   10314      0.3421
 25.0000    50.0000   35450      0.2346
 50.0000    75.0000   53238      0.1649
 75.0000   100.0000   68805      0.1228
100.0000   150.0000  176025      0.0894
150.0000   200.0000  217585      0.0587
200.0000   300.0000  531651      0.0272
300.0000   400.0000  609990      0.0072
400.0000   600.0000 1297109     -0.0212
600.0000   800.0000 1383337     -0.0102
800.0000 1,000.0000 1378747     -0.0145


### Parte 16 (adendo): estabilidade do corte de Conley em 2022

Regra adicionada em 2026-11-09, depois dos primeiros resultados do notebook.

O corte de 2026 vem de uma só eleição. Ajusta-se a especificação final no 2º turno de 2022 (`MAPA_FIT_2022`, defasagem = 2018; com colunas `w_*` recalculadas nos dados de 2022 se o final tiver SLX) e aplica-se a mesma regra ao correlograma. Se 0,5·CORTE ≤ CORTE_2022 ≤ 2·CORTE, o CORTE de 2026 fica; senão CORTE = max(CORTE, CORTE_2022). Depois, a curva do EP contra o corte (50 a 800 km) para o modelo final em 2026. A Parte 16 (tabela de EPs) e a Parte 17 usam o CORTE final.

In [11]:
from src import backtest as BT
CORTE_2026 = CORTE
M22 = BT.MAPA_FIT_2022
df22, m22 = (S.adicionar_slx(df, M22, list(CF.termos.slx), W) if CF.termos.slx else (df, M22))
y22, n22 = df22[M22["y"]].to_numpy(float), df22[M22["n"]].to_numpy(float)
A22 = CF.ajustar(df22, m22)
PI22 = np.asarray(CF.prever(A22, df22, m22), float)
R22 = G.residuos_quantilicos_rapido(y22, n22, PI22, theta=E.prever_theta(A22, df22, m22), rng=0)
CG22 = S.correlograma(R22, lat, lon, FAIXAS); CG22.to_csv(TAB / "final_correlograma_2022.csv", index=False)
CORTE_2022 = corte_de(CG22)
if 0.5 * CORTE_2026 <= CORTE_2022 <= 2 * CORTE_2026:
    CORTE = CORTE_2026; MOTIVO_CORTE = f"CORTE_2022={CORTE_2022:g} km dentro de [{0.5*CORTE_2026:g}, {2*CORTE_2026:g}]; corte de 2026 mantido"
else:
    CORTE = max(CORTE_2026, CORTE_2022); MOTIVO_CORTE = f"CORTE_2022={CORTE_2022:g} km fora de [{0.5*CORTE_2026:g}, {2*CORTE_2026:g}]; usa o máximo"
print(CG22.to_string(index=False)); print(f"corte 2026 = {CORTE_2026:g} km | corte 2022 = {CORTE_2022:g} km | corte final = {CORTE:g} km |", MOTIVO_CORTE)
nomes_c = list(AF.nomes); mod_c = AF.modelo; ep_mod = np.sqrt(np.diag(np.asarray(mod_c.cov_params(), float)))
LC = []
for ck in [50, 100, 150, 200, 250, 300, 400, 500, 600, 800]:
    e = np.sqrt(np.diag(np.asarray(S.cov_conley(mod_c, lat, lon, ck), float)))
    for j, nm in enumerate(nomes_c): LC.append(dict(param=nm, coef=float(mod_c.params[j]), corte_km=ck, ep=float(e[j]), razao=float(e[j] / ep_mod[j])))
CURVA = pd.DataFrame(LC); CURVA.to_csv(TAB / "final_ep_corte.csv", index=False)
print(CURVA.pivot(index="param", columns="corte_km", values="razao").round(2).to_string())
FIN.update({"corte_2026": CORTE_2026, "corte_2022": CORTE_2022, "corte_final": CORTE, "motivo_corte": MOTIVO_CORTE})
(TAB / "final_decisao.json").write_text(json.dumps(FIN, indent=2, ensure_ascii=False, default=str))

  ini_km     fim_km   pares  correlacao
  0.0000    10.0000     617      0.4760
 10.0000    25.0000   10314      0.4061
 25.0000    50.0000   35450      0.2879
 50.0000    75.0000   53238      0.2192
 75.0000   100.0000   68805      0.1755
100.0000   150.0000  176025      0.1165
150.0000   200.0000  217585      0.0839
200.0000   300.0000  531651      0.0299
300.0000   400.0000  609990     -0.0331
400.0000   600.0000 1297109     -0.0350
600.0000   800.0000 1383337      0.0022
800.0000 1,000.0000 1378747     -0.0133
corte 2026 = 200 km | corte 2022 = 200 km | corte final = 200 km | CORTE_2022=200 km dentro de [100, 400]; corte de 2026 mantido


corte_km                 50     100    150    200    250    300    400    500    600    800
param                                                                                      
bloco_esquerda        1.1400 1.2500 1.3400 1.4100 1.5100 1.6200 1.8200 1.9700 2.0700 2.2300
bloco_outro           0.9800 0.9700 0.9500 0.9500 0.9600 0.9500 0.9000 0.8800 0.8800 0.8700
cobertura_pbf         1.0300 1.0900 1.1800 1.2600 1.3300 1.4000 1.5400 1.6800 1.7800 1.8900
comparecimento        1.0300 1.0400 1.0900 1.1600 1.2500 1.3400 1.5100 1.6400 1.7300 1.8300
const                 1.4000 1.7900 2.0300 2.1500 2.2100 2.2400 2.2400 2.1800 2.1000 1.9300
disp_const            1.4300 1.8100 2.1300 2.3600 2.5200 2.6000 2.6600 2.6800 2.7600 2.8100
disp_log_n            1.3300 1.4900 1.6100 1.7400 1.8600 1.9700 2.0800 2.1300 2.1900 2.3000
disp_reg_Centro-Oeste 1.2300 1.4400 1.6100 1.7700 1.9000 2.0000 2.1400 2.2100 2.2500 2.2400
disp_reg_Nordeste     1.4600 1.7600 2.0300 2.2000 2.3100 2.3600 2.4100 2.4500 2.

6768

**Leitura.** O correlograma de 2022 cai abaixo de 0,05 no mesmo ponto que o de 2026, e o corte de 2022 é 200 km, dentro de [100, 400]. O corte de 2026 é mantido. A correlação de 2022 é um pouco maior a curta distância (0,48 contra 0,40 até 10 km), mas o alcance é o mesmo.

*Números da execução de 2026-11-09; se o notebook for reexecutado, confira as saídas acima.*

In [12]:
mod = AF.modelo
nomes_p = list(AF.nomes); assert len(nomes_p) == len(mod.params), (len(nomes_p), len(mod.params))
def ep(cov): return np.sqrt(np.diag(np.asarray(cov, float)))
EP = {"ep_modelo": ep(mod.cov_params())}
for tag, f in [("ep_conley_05", 0.5), ("ep_conley_1", 1.0), ("ep_conley_2", 2.0)]:
    EP[tag] = ep(S.cov_conley(mod, lat, lon, CORTE * f))
EP["ep_cluster_uf"] = ep(S.cov_cluster(mod, uf, correcao=True))
TB = pd.DataFrame({"param": nomes_p, "tipo": ["beta"] * NB_ + ["gamma"] * (len(nomes_p) - NB_), "coef": np.asarray(mod.params, float), **EP})
TB["razao_conley_modelo"] = TB.ep_conley_1 / TB.ep_modelo
TB["signif_modelo"] = (TB.coef / TB.ep_modelo).abs() > 1.96
TB["signif_conley"] = (TB.coef / TB.ep_conley_1).abs() > 1.96
TB["muda"] = TB.signif_modelo != TB.signif_conley
TB.to_csv(TAB / "final_ep.csv", index=False)
print(f"corte {CORTE:g} km (×0,5 = {CORTE*.5:g}; ×2 = {CORTE*2:g})")
print(TB.to_string(index=False))
rb = TB[TB.tipo == "beta"].razao_conley_modelo
print(f"\nrazão Conley/modelo (β): mediana {rb.median():.2f} | mín {rb.min():.2f} | máx {rb.max():.2f}")
print("coeficientes que mudam de significância:", list(TB[TB.muda].param) or "nenhum")

corte 200 km (×0,5 = 100; ×2 = 400)
                param  tipo    coef  ep_modelo  ep_conley_05  ep_conley_1  ep_conley_2  ep_cluster_uf  razao_conley_modelo  signif_modelo  signif_conley  muda
                const  beta  0.1947     0.0031        0.0055       0.0066       0.0069         0.0078               2.1528           True           True False
                  lag  beta  0.7760     0.0041        0.0046       0.0051       0.0059         0.0070               1.2652           True           True False
          pct_dep_esq  beta  0.0106     0.0017        0.0019       0.0021       0.0027         0.0027               1.2565           True           True False
       comparecimento  beta -0.0187     0.0021        0.0021       0.0024       0.0031         0.0030               1.1570           True           True False
             log_dens  beta  0.0150     0.0028        0.0030       0.0035       0.0042         0.0043               1.2221           True           True False
         p

**Leitura.** `razao_conley_modelo` > 1 indica que o EP do modelo subestima a incerteza. O corte foi fixado
pela regra, antes de ver os EPs; as colunas ×0,5 e ×2 e o conglomerado por UF mostram a sensibilidade.

No modelo final, a razão Conley/modelo dos β tem mediana 1,69 (de 0,95 a 3,52); as maiores são as dummies de região (Sul 3,52; Centro-Oeste 3,12). Cinco coeficientes perdem significância com Conley: `w_pct_vab_agro`, `bloco_esquerda`, `reg_Centro-Oeste`, `reg_Norte` e `disp_reg_Sul`. As conclusões sobre esses termos devem usar o EP de Conley.

*Números da execução de 2026-11-09; se o notebook for reexecutado, confira as saídas acima.*

## Parte 17 · Estimativas regionais e por UF do modelo final (LOSO)

**Procedimento** (o mesmo do notebook 03, Parte 12). Para cada UF, o modelo final é reajustado sem ela e a prevê.
Intervalos: B = 1.000 sorteios por UF retida (semente 7, quantis 2,5/50/97,5 %; guardamos também 5/95 %) de
(i) β ~ N(β̂, Σ̂), com Σ̂ o bloco de β da covariância do ajuste sem a UF; (ii) choque de UF δ ~ N(0, σ²) no logit;
(iii) votos ~ BetaBin(n, π, θᵢ) com **θᵢ por município** (`prever_theta`, dispersão por região e log n).

**Duas covariâncias** são comparadas: a **Conley** (corte da Parte 16, calculada no ajuste de treino de cada UF)
e a **do modelo**. **Dois σ**: 0 e σ_UF = RMS dos resíduos LOSO das UFs no logit (receita do notebook 03).

**Aviso.** σ_UF é estimado nas mesmas 27 UFs em que a cobertura é conferida, então a cobertura com σ_UF é
**otimista**. A cobertura com σ = 0 não usa essa informação, mas ignora o erro de extrapolar para uma UF inédita.

In [13]:
from scipy.special import logit
def parametros_f(nome_uf):
    tr = np.flatnonzero(uf != nome_uf); te = np.flatnonzero(uf == nome_uf)
    a = CF.ajustar(df.iloc[tr], mapa)
    X = E.montar_X(df.iloc[te], mapa, a.pad, CF.termos)
    nb_ = a.extra["n_beta"]; assert X.shape[1] == nb_
    th = E.prever_theta(a, df.iloc[te], mapa)
    cm = np.asarray(a.modelo.cov_params(), float)[:nb_, :nb_]
    cc = np.asarray(S.cov_conley(a.modelo, lat[tr], lon[tr], CORTE), float)[:nb_, :nb_]
    return dict(beta=np.asarray(a.modelo.params, float)[:nb_], cov_modelo=cm, cov_conley=cc, X=X.to_numpy(float), theta=th, te=te)
t0 = time.time()
with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
    PARF = dict(zip(UFS_, ex.map(parametros_f, UFS_)))
print("ajustes LOSO do modelo final:", NOME_F, round(time.time() - t0), "s")
pi_loso_f = PIL[NOME_F]
prev_uf_f = M.agregar(pi_loso_f, n, uf).drop("Brasil")
res_logit_f = logit(prev_uf_f[UFS_].values) - logit(obs_uf[UFS_].values)
SIG_F = float(np.sqrt(np.mean(res_logit_f ** 2)))
print(f"σ_UF (RMS dos resíduos LOSO no logit) = {SIG_F:.4f} | viés médio = {res_logit_f.mean():+.4f}")
print("razão média diag(cov Conley)/diag(cov modelo) do bloco β, por UF retida: mediana",
      round(float(np.median([np.mean(np.diag(v['cov_conley']) / np.diag(v['cov_modelo'])) for v in PARF.values()])), 2))

ajustes LOSO do modelo final: bb_slx 46 s
σ_UF (RMS dos resíduos LOSO no logit) = 0.0523 | viés médio = -0.0187
razão média diag(cov Conley)/diag(cov modelo) do bloco β, por UF retida: mediana 3.04


In [14]:
n_uf = pd.Series(n).groupby(uf).sum()[UFS_].to_numpy(); uf_reg = pd.Series(reg).groupby(uf).first()[UFS_].to_numpy()
def simular_f(chave, sigma, B=1000, seed=7):
    rng = np.random.default_rng(seed)
    return np.column_stack([M.simular_votos_grupo(v["beta"], v[chave], v["X"], n[v["te"]], np.zeros(len(v["te"])), link=E.LIGACOES[CF.ligacao],
                            theta=v["theta"], sigma=sigma, B=B, rng=rng)[:, 0] for v in PARF.values()])
def resumo_f(chave, sigma):
    V = simular_f(chave, sigma); linhas = {}
    for i, u in enumerate(UFS_): linhas[("uf", u)] = (V[:, i], n_uf[i], obs_uf[u])
    for r in REGS:
        m = uf_reg == r; linhas[("regiao", r)] = (V[:, m].sum(1), n_uf[m].sum(), obs_reg[r])
    linhas[("regiao", "Brasil")] = (V.sum(1), n_uf.sum(), y.sum() / n.sum())
    out = []
    for (nivel, nome), (v, nn, real) in linhas.items():
        q = np.quantile(v / nn, [0.025, 0.05, 0.5, 0.95, 0.975])
        pt = (prev_uf_f[UFS_].values * n_uf)[uf_reg == nome].sum() / nn if nivel == "regiao" and nome != "Brasil" else \
             (prev_uf_f[nome] if nivel == "uf" else (prev_uf_f[UFS_].values * n_uf).sum() / nn)
        out.append(dict(nivel=nivel, nome=nome, lula_pct=pt * 100, q025=q[0] * 100, q05=q[1] * 100, q50=q[2] * 100, q95=q[3] * 100, q975=q[4] * 100, real_pct=real * 100))
    d = pd.DataFrame(out); d["erro_pp"] = d.lula_pct - d.real_pct
    d["dentro_ic95"] = (d.real_pct >= d.q025) & (d.real_pct <= d.q975); d["dentro_ic90"] = (d.real_pct >= d.q05) & (d.real_pct <= d.q95)
    d["largura_pp"] = d.q975 - d.q025; d["largura90_pp"] = d.q95 - d.q05
    return d
VARS = {("conley", "sigma_uf"): ("cov_conley", SIG_F), ("conley", "zero"): ("cov_conley", 0.0),
        ("modelo", "sigma_uf"): ("cov_modelo", SIG_F), ("modelo", "zero"): ("cov_modelo", 0.0)}
EST_F = pd.concat([resumo_f(ch, s).assign(covariancia=cv, sigma=sg) for (cv, sg), (ch, s) in VARS.items()], ignore_index=True)
col0 = ["covariancia", "sigma"]
EST_F[EST_F.nivel == "regiao"].drop(columns="nivel").rename(columns={"nome": "regiao"}).to_csv(TAB / "final_estimativas_regionais.csv", index=False)
EST_F[EST_F.nivel == "uf"].drop(columns="nivel").rename(columns={"nome": "uf"}).to_csv(TAB / "final_estimativas_uf.csv", index=False)
COB = []
for (cv, sg), d in EST_F.groupby(col0):
    u, r = d[d.nivel == "uf"], d[(d.nivel == "regiao") & (d.nome != "Brasil")]
    COB.append(dict(covariancia=cv, sigma=sg, valor_sigma=SIG_F if sg == "sigma_uf" else 0.0, cob95_uf=u.dentro_ic95.mean(), n_uf_dentro95=int(u.dentro_ic95.sum()),
                    cob90_uf=u.dentro_ic90.mean(), cob95_regiao=r.dentro_ic95.mean(), n_reg_dentro95=int(r.dentro_ic95.sum()), cob90_regiao=r.dentro_ic90.mean(),
                    largura95_uf_pp=u.largura_pp.mean(), largura95_reg_pp=r.largura_pp.mean()))
COB = pd.DataFrame(COB); COB.to_csv(TAB / "final_cobertura.csv", index=False)
print(COB.to_string(index=False)); print("(nominal 95 % e 90 %; 27 UFs, 5 regiões; cobertura com σ_UF é otimista)")
PF = EST_F[(EST_F.covariancia == "conley") & (EST_F.sigma == "sigma_uf") & (EST_F.nivel == "regiao")]
print(PF[["nome", "lula_pct", "q05", "q95", "q025", "q975", "real_pct", "erro_pp", "dentro_ic95"]].to_string(index=False))

covariancia    sigma  valor_sigma  cob95_uf  n_uf_dentro95  cob90_uf  cob95_regiao  n_reg_dentro95  cob90_regiao  largura95_uf_pp  largura95_reg_pp
     conley sigma_uf       0.0523    1.0000             27    0.9630        1.0000               5        1.0000           5.0762            2.6456
     conley     zero       0.0000    0.5926             16    0.4815        0.6000               3        0.6000           2.0691            0.8771
     modelo sigma_uf       0.0523    1.0000             27    0.9630        1.0000               5        1.0000           4.9304            2.6300
     modelo     zero       0.0000    0.4444             12    0.2963        0.4000               2        0.4000           1.7154            0.6785
(nominal 95 % e 90 %; 27 UFs, 5 regiões; cobertura com σ_UF é otimista)
        nome  lula_pct     q05     q95    q025    q975  real_pct  erro_pp  dentro_ic95
Centro-Oeste   40.0417 38.8480 41.2693 38.6388 41.5124   39.9912   0.0504         True
    Nordeste  

In [15]:
bb = pd.read_csv(TAB / "estimativas_regionais.csv").set_index("regiao")
pf = PF.set_index("nome")
CMP = pd.DataFrame({"real_pct": pf.real_pct,
    "bb_backward_pct": bb.lula_pct, "bb_backward_q025": bb.q025, "bb_backward_q975": bb.q975, "bb_backward_erro_pp": bb.erro_pp, "bb_backward_largura_pp": bb.largura_pp,
    "final_pct": pf.lula_pct, "final_q025": pf.q025, "final_q975": pf.q975, "final_q05": pf.q05, "final_q95": pf.q95, "final_erro_pp": pf.erro_pp, "final_largura_pp": pf.largura_pp}).rename_axis("regiao")
CMP.to_csv(TAB / "final_comparacao_regional.csv"); print(CMP.round(3).to_string())
print("erro absoluto médio nas 5 regiões (pp): bb_backward", round(CMP.drop("Brasil").bb_backward_erro_pp.abs().mean(), 3), "| final", round(CMP.drop("Brasil").final_erro_pp.abs().mean(), 3))
d = PF.sort_values("lula_pct"); b2 = bb.loc[d.nome]
fig = go.Figure()
fig.add_trace(go.Scatter(x=b2.lula_pct, y=d.nome, mode="markers", name="bb_backward (IC95)", error_x=dict(type="data", symmetric=False, array=b2.q975 - b2.lula_pct, arrayminus=b2.lula_pct - b2.q025), marker=dict(color="#888", size=8)))
fig.add_trace(go.Scatter(x=d.lula_pct, y=d.nome, mode="markers", name=f"{NOME_F}, Conley (IC95)", error_x=dict(type="data", symmetric=False, array=d.q975 - d.lula_pct, arrayminus=d.lula_pct - d.q025), marker=dict(color="#1b6ca8", size=9)))
fig.add_trace(go.Scatter(x=d.real_pct, y=d.nome, mode="markers", name="real", marker=dict(color="#c2571a", symbol="diamond", size=10)))
fig.update_layout(title="Share de Lula por região (LOSO, σ_UF): bb_backward e modelo final", xaxis_title="% dos votos válidos", template="plotly_white", height=460)
G.salvar(fig, FIG / "final_regional.html"); fig.show()
print("tempo total do notebook:", round(time.time() - T0), "s")

              real_pct  bb_backward_pct  bb_backward_q025  bb_backward_q975  bb_backward_erro_pp  bb_backward_largura_pp  final_pct  final_q025  final_q975  final_q05  final_q95  final_erro_pp  final_largura_pp
regiao                                                                                                                                                                                                            
Centro-Oeste   39.9910          40.3230           38.6130           41.9710               0.3320                  3.3580    40.0420     38.6390     41.5120    38.8480    41.2690         0.0500            2.8740
Nordeste       69.3690          69.4150           68.5030           70.2660               0.0470                  1.7640    69.4110     68.5380     70.2210    68.6980    70.0980         0.0420            1.6830
Norte          50.0050          49.4970           48.1020           51.0270              -0.5080                  2.9260    49.5220     48.0810     50.9030 

tempo total do notebook: 294 s


**Leitura.** O modelo final erra as regiões em 0,32 p.p. em média (0,36 em `bb_backward`), e o Brasil em 0,17 p.p. (51,95% contra 51,78% real). Os maiores erros são Sudeste (+0,62) e Norte (−0,48). Os intervalos ficam um pouco mais estreitos que os de `bb_backward` (Centro-Oeste 2,87 contra 3,36 p.p.). Com 5 regiões, a comparação é descrição, não evidência de superioridade ([doc 06](../docs/pressupostos/06-estimativas-regionais.md), seção 5). A cobertura (27/27 UFs com σ_UF) é otimista, porque σ_UF é ajustado nas mesmas UFs.

*Números da execução de 2026-11-09; se o notebook for reexecutado, confira as saídas acima.*